# 03 Train Legal LoRA Adapter

This notebook trains the `legal` standalone PEFT LoRA adapter and writes it to `adapters/legal/`.

```mermaid
flowchart LR
    A[datasets/generated/legal.jsonl] --> B[Qwen chat template]
    B --> C[PEFT LoRA training]
    C --> D[adapters/legal/]
    C --> E[MLflow run]
```

The adapter is not merged into the base model. vLLM will load it later as a runtime LoRA adapter.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PROJECT_SPEC.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from llmops_demo.settings import settings, ensure_dirs

cfg = settings()
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {cfg.base_model}")
print(f"Adapters: {cfg.adapters}")

## Preflight

Expected inputs:

- `datasets/generated/legal.jsonl`
- access to `BASE_MODEL from .env`
- CUDA GPU recommended for practical runtime

In [ ]:
dataset_path = cfg.dataset_dir / "legal.jsonl"
adapter_path = cfg.adapter_dir / "legal"
print(f"Dataset exists: {dataset_path.exists()} - {dataset_path}")
print(f"Adapter output: {adapter_path}")

## Train

This calls the shared production training entry point, logs the run to MLflow, and saves the standalone PEFT adapter.

In [ ]:
from training.train_lora import train_adapter

train_adapter("legal", cfg)

## Verify adapter files

Expected output: `adapter_config.json`, adapter weights, tokenizer files, and related PEFT metadata under `adapters/legal/`.

In [ ]:
for path in sorted((cfg.adapter_dir / "legal").glob("*")):
    print(path)